# 품질 검수 — 양성판정·VLM 정책 검토 노트북

`worker/validation/quality_check_e2e_validation.ipynb`(이하 e2e 노트북)는 **콜드/웜 latency와
BE로 나가는 최종 JSON 모양**을 빠르게 확인하는 스모크 테스트용이다. 이 노트북은 별도로 둔다 —
아래 5가지는 매 검증마다 도는 게 아니라 **정책을 바꿀 때만** 필요한 무거운 분석이라, e2e
노트북에 섞으면 그 노트북의 원래 목적(빠른 회귀 확인)이 흐려진다.

다루는 5가지, 그리고 실행에 GPU가 필요한지:

| 섹션 | 내용 | GPU 필요 여부 |
| --- | --- | --- |
| A | 하자 프롬프트 문구 실험 | **필요** (SAM3 실호출) |
| B | 양성판정(하자 신뢰도) 고도화 — 부위별 스코프 제안 | 불필요 (기존 로그 분석) |
| C | 탐지 범위 파악 — 카테고리·하자유형 커버리지 | 불필요 (레지스트리 조회) |
| D | VLM 과다 호출 방지 정책 재검토 | 불필요 (기존 로그 분석) |
| E | 탐지결과·bbox FE 전달용 그룹핑 정책 | 불필요 (순수 로직) |

**B/C/D/E는 이미 한 번 실행해서 실제 결과를 출력 셀에 남겨 뒀다** — `worker/logs/quality/`에
쌓인 프로덕션 로그(2026-08-07, sam3_pcs 백엔드 17건)와 `worker/models/sam3/registry.py`를
그대로 읽은 결과다. 로그가 더 쌓이면 재실행해서 갱신한다. **A만 스켈레톤이다** — SAM3를 실제로
호출해야 해서 GPU 워커 환경(`QUALITY_BACKEND=sam3_pcs`, HF gated 토큰)에서 실행해야 한다.

⚠️ **여기서 나온 제안(특히 B/D)은 분석·가설 단계다.** `worker/models/sam3/registry.py`·
`quality_vlm/describer.py`처럼 실제 정책 코드를 바꾸는 건 이 노트북의 범위 밖이다 — 루트
`CLAUDE.md`의 "새 도메인 로직은 `docs/plans/`에 계획부터" 규칙대로, 여기서 확인된 근거를 들고
별도 계획 문서를 먼저 쓴 뒤에 진행한다.


In [ ]:
import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

_REPO_ROOT = Path.cwd().resolve().parents[1]  # worker/validation/ -> worker/ -> repo root
_WORKER_MODELS = _REPO_ROOT / "worker" / "models"
sys.path.insert(0, str(_WORKER_MODELS))

LOG_DIR = _REPO_ROOT / "worker" / "logs" / "quality"
LOG_FILES = sorted(LOG_DIR.glob("*.jsonl"))
print(f"로그 파일: {[p.name for p in LOG_FILES]}")

def load_jobs():
    jobs = []
    for path in LOG_FILES:
        with open(path, encoding="utf-8") as f:
            jobs.extend(json.loads(line) for line in f if line.strip())
    return jobs

JOBS = load_jobs()
SAM3_JOBS = [j for j in JOBS if j.get("model", {}).get("backend") == "sam3_pcs"]
print(f"전체 job {len(JOBS)}건, 그중 실모델(sam3_pcs) {len(SAM3_JOBS)}건")

## A. 하자 프롬프트 문구 실험

**왜**: `prompts.py`의 `_DEFECT_PROMPTS`는 LAB-12/LAB-46 스파이크(`AI/validation/QUAL/
sam3-condition-check/`)에서 나온 문구를 그대로 쓴다. hole_tear 프롬프트("a hole or tear in
fabric")는 그 스파이크에서 이미 "구조적 개구부를 구멍으로 오인"한다고 결론 났고, 지금 프로덕션
로그(B절)도 그 결론과 일치한다 — **문구를 안 바꾸면 이 오탐은 그대로 남는다.**

**시도해 볼 변형**(SAM3는 짧은 명사구에만 반응한다 — `prompts.py` 모듈 docstring 참고, 문장으로
늘이면 반응이 약해짐이 실측됨):

- `"a hole in fabric"` / `"a tear in fabric"` — 지금 하나로 합쳐진 프롬프트를 분리했을 때
  구조적 개구부 오탐 비율이 갈리는지(구멍과 찢어짐은 원인이 달라 SAM3 반응도 다를 수 있음).
- `"a damaged hole"` / `"a torn opening"` — "damaged"/"torn"을 넣어 "원래 있는 개구부"와
  구분되는 신호를 주려는 시도. 다만 SAM3가 형용사를 얼마나 반영하는지는 미검증.
- 부정 프롬프트 대신 이미 있는 `STRUCTURAL_BENIGN_PROMPTS`(`"an armhole"`/`"a neckline"`/
  `"a hem"`/`"a cuff"`) 자체의 표현이 실제 이미지의 그 부위를 얼마나 잘 잡는지 재확인 —
  benign 프롬프트가 놓치면 억제가 안 걸려 hole_tear 오탐이 그대로 살아남는다(B절의
  COLLAR 사례가 이 실패 가능성을 보여준다: confidence 0.3~0.5로 낮게라도 11건이 여전히
  잡혔다는 건 neckline benign 프롬프트가 완전히 다 걸러내지 못했거나, 애초에 진짜 애매한
  경계라는 뜻).

**방법**: 아래 `PROMPT_CANDIDATES`에 실제 QUAL 스파이크 테스트 이미지(`AI/validation/QUAL/
qc_shared.py`가 참조하는 샘플들, 또는 최근 프로덕션 job의 `image_urls`)를 넣고 각 변형으로
`segmenter.segment()`를 돌려 confidence·인스턴스 수를 비교한다. **GPU 필요** — 로컬 워커
환경(`QUALITY_BACKEND=sam3_pcs`)에서 실행.

In [ ]:
# TODO(GPU 환경에서 실행): 실제 이미지 URL로 채운 뒤 실행
PROMPT_CANDIDATES = {
    "baseline": "a hole or tear in fabric",  # 현재 프로덕션 프롬프트
    "split_hole": "a hole in fabric",
    "split_tear": "a tear in fabric",
    "damaged_hole": "a damaged hole",
    "torn_opening": "a torn opening",
}

TEST_IMAGE_URLS = [
    # "https://i15a108.p.ssafy.io/vintib-public/product-images/78/....jpg",  # CHEST_CENTER 실제 검출 사례
    # "https://i15a108.p.ssafy.io/vintib-public/product-images/81/....jpg",  # COLLAR 낮은 confidence 사례
]

# from sam3.segmenter import build_segmenter
# from sam3 import registry as sam3_registry
#
# segmenter = build_segmenter("sam3_pcs")
# for name, prompt in PROMPT_CANDIDATES.items():
#     for url in TEST_IMAGE_URLS:
#         # segmenter 내부 API로 단일 프롬프트 호출 — segmenter.py의 실제 시그니처 확인 후 채울 것
#         ...
#         print(name, url, "-> confidence=?, instances=?")
print("GPU 워커 환경에서 주석 해제 후 실행")

## B. 양성판정(하자 신뢰도) 고도화 — 부위별 스코프 제안

**현재 정책**(`sam3/findings.py:_status_of`, `sam3/registry.py:DEFECT_TAXONOMY`): `hole_tear`는
`trusted=False`라 **부위·confidence와 무관하게 항상 SUSPECTED**다. `suppress_benign()`이
암홀·넥라인·밑단·소매끝과 겹치는 인스턴스를 먼저 걸러내지만, 그 필터를 통과한 뒤에도 한 번 더
전부 SUSPECTED로 깎인다 — 두 안전장치가 겹쳐 있다.

**가설**: LAB-46의 오탐 원인은 "구조적 개구부(넥라인·밑단·소매끝·암홀)를 구멍으로 오인"하는
것이었다. 이 개구부들은 3×3 그리드에서 `COLLAR`/`HEM_*`/`SLEEVE_*`/어깨 인접 구역에 몰려 있고,
**`CHEST_CENTER`처럼 옷에 자연적인 개구부가 없는 구역**은 이 혼동 원인 자체가 없다. 즉
`trusted` 판단을 하자 유형 단위가 아니라 **(하자 유형, 부위) 조합 단위**로 내리면, 오탐 위험이
낮은 부위에서는 confidence가 높을 때 DETECTED로 승격시킬 수 있다.

**아래는 실제 프로덕션 로그로 이 가설을 검증한 결과다** — 재실행하면 최신 로그로 갱신된다.

In [ ]:
from collections import Counter, defaultdict

hole_tear_by_section = defaultdict(list)
for job in SAM3_JOBS:
    for finding in job.get("findings", []):
        if finding["defect_type"] == "HOLE_TEAR":
            hole_tear_by_section[finding["section"]].append(
                (finding["status"], finding["confidence"])
            )

CONFIDENT_THRESHOLD = 0.55  # sam3/registry.py의 confident_threshold

print(f"{'부위':15s} {'건수':>4s} {'상태':>12s} {'confidence 범위':>20s} {'threshold 이상':>14s}")
for section, items in sorted(hole_tear_by_section.items()):
    confs = [c for _, c in items]
    statuses = Counter(s for s, _ in items)
    above = sum(1 for c in confs if c >= CONFIDENT_THRESHOLD)
    print(
        f"{section:15s} {len(items):>4d} {dict(statuses)!s:>12s} "
        f"({min(confs):.3f}~{max(confs):.3f}) {above:>14d}/{len(items)}"
    )


부위              건수           상태           confidence 범위 threshold 이상
CHEST_CENTER        10 {'SUSPECTED': 10} (0.781~0.817)             10/10
COLLAR               11 {'SUSPECTED': 11} (0.309~0.499)              0/11
HEM_CENTER            1 {'SUSPECTED': 1} (0.696~0.696)              1/1
HEM_RIGHT             4 {'SUSPECTED': 4} (0.688~0.688)              4/4


**읽는 법**: `CHEST_CENTER`의 hole_tear 10건은 전부 confidence 0.78~0.82로 threshold(0.55)를
훌쩍 넘지만, `trusted=False`라 전부 SUSPECTED로 깎여 있다. 반대로 `HEM_CENTER`/`HEM_RIGHT`도
confidence는 threshold를 넘지만(0.688~0.696), 이 부위는 LAB-46이 지적한 "밑단" 구조적 개구부와
정확히 겹친다 — 여기서 승격시키면 LAB-46이 잡아낸 오탐 패턴을 그대로 재현할 위험이 크다.
`COLLAR`는 애초에 confidence가 threshold 미달(0.31~0.50)이라 지금 정책이든 부위별 정책이든
결과가 같다.

**제안(가설 단계, 정책 코드는 아직 안 바꿈)**:

1. `DEFECT_TAXONOMY`의 `trusted: bool` 하나를, 하자 유형별로 **"신뢰 제외 부위 집합"**
   (예: hole_tear → `{COLLAR, HEM_LEFT, HEM_CENTER, HEM_RIGHT, SLEEVE_LEFT, SLEEVE_RIGHT}`)으로
   바꾼다. 그 집합에 없는 부위(`CHEST_CENTER`, `SHOULDER_LEFT`, `SHOULDER_RIGHT`)에서는 기존
   `confident_threshold` 로직(0.55 이상 → DETECTED)을 그대로 적용한다.
2. **`CHEST_CENTER`가 정말 안전한 구역인지**는 이 10건만으로 확정하기엔 표본이 작다(전부 같은
   며칠 사이 테스트 데이터). 실제 셀러 소명(`needs_seller_review`)이 쌓이면 그 결과로
   재검증해야 한다 — 이게 아래 D절/그리고 "detected 컬럼 보완" 논의와 이어진다.
3. `HEM_*`처럼 threshold를 넘는데도 부위상 여전히 못 믿는 경우가 있다는 게 확인됐으니, **부위
   스코프만으로 다 해결되진 않는다** — `suppress_benign()`의 겹침 판정(IoU/containment)이
   왜 이 4건을 못 걸러냈는지도 같이 봐야 한다(benign 프롬프트 반응이 약했을 수 있음, A절의
   프롬프트 실험과 연결).

**"detected 컬럼" 설계 의도 보완 아이디어**: 지금 `DETECTED`는 순수하게 "모델이 그 유형·부위를
얼마나 믿는가"다. 사용자가 짐작한 "사용자가 하자 아니라고 안 한 상태"는 지금 코드엔 없다 — 그런
사람 피드백 루프 자체가 없다(BE가 `statement_required` 응답을 저장은 하지만 AI가 다시 읽어서
정책을 조정하는 경로가 없음). 이게 바로 모든 문서(CH006/CH028/registry.py 주석)가 반복하는
"라벨링된 검증셋이 없다"는 한계의 근본 원인이다 — **셀러가 SUSPECTED 항목에 대해 "실제 하자
맞음/아님"으로 응답한 걸 AI 쪽에 다시 흘려주는 파이프라인**이 있으면, 그게 곧 지금 없는
라벨셋이 된다. 이건 BE 쪽 저장 스키마가 관여하는 변경이라 `docs/requests/BE/`에 요청 문서를
남기는 게 맞다(추후 계획 문서에서 다룰 것).

## C. 탐지 범위 파악 — 카테고리·하자유형 커버리지

**왜**: "SAM3가 의류 데이터가 없어서 성능이 안 좋은가?"라는 질문에 답하려면, 먼저 **어디까지
검증됐고 어디부터 미검증인지**를 갈라야 한다. SAM3 자체는 의류 전용으로 훈련된 모델이 아니라
Meta의 범용 promptable segmentation 모델이다(zero-shot) — `registry.py` 주석이 이미 이걸
인정한다: stain/dirt_mark/hole_tear 세 유형만 LAB-12/LAB-46으로 실측했고, discoloration/
pilling/fray 세 유형은 "SAM3 스파이크 대상이 아니었다"(=한 번도 검증 안 됨). 카테고리는
6종(top_knit/top_shirt/outer_jacket/outer_coat/bottom_pants/bottom_skirt) 전부 프롬프트셋이
있지만, **실제 프로덕션 로그에 등장한 카테고리는 top_knit/top_shirt/outer_jacket 3종뿐**이다
(bottom_pants/bottom_skirt/outer_coat는 코드는 있지만 실환경 검증 0건).

In [ ]:
from sam3.contracts import GarmentCategory, BaseColor, DefectType
from sam3 import registry as sam3_registry
from sam3.prompts import select_prompt_set, SECTION_LABELS

print("--- 카테고리 x 색상 -> 프롬프트셋 (구조적 커버리지) ---")
for cat in GarmentCategory:
    ps = select_prompt_set(cat, BaseColor.light)
    print(f"  {cat.value:15s} -> {ps.key:25s} (부위 {len(SECTION_LABELS[cat])}개)")

print()
print("--- 실환경 검증 여부 (프로덕션 로그 기준) ---")
seen_categories = Counter(j["category"] for j in SAM3_JOBS)
for cat in GarmentCategory:
    n = seen_categories.get(cat.value, 0)
    mark = "실사용됨" if n else "코드만 있음, 실사용 0건"
    print(f"  {cat.value:15s} {n:>3d}건  ({mark})")

print()
print("--- DEFECT_TAXONOMY: 검증 여부 ---")
for dt in DefectType:
    pol = sam3_registry.defect_policy(dt)
    verified = "실측 검증됨" if "실측" in pol["evidence"] else "미검증"
    print(f"  {dt.value:15s} trusted={pol['trusted']!s:5s} [{verified}] {pol['evidence']}")


--- 카테고리 x 색상 -> 프롬프트셋 (구조적 커버리지) ---
  TOP_KNIT        -> top_knit/light           (부위 9개)
  TOP_SHIRT       -> top_shirt/light          (부위 9개)
  OUTER_JACKET    -> outer_jacket/light       (부위 9개)
  OUTER_COAT      -> outer_coat/light         (부위 9개)
  BOTTOM_PANTS    -> bottom_pants/light       (부위 9개)
  BOTTOM_SKIRT    -> bottom_skirt/light       (부위 9개)

--- 실환경 검증 여부 (프로덕션 로그 기준) ---
  TOP_KNIT          6건  (실사용됨)
  TOP_SHIRT        11건  (실사용됨)
  OUTER_JACKET      0건  (코드만 있음, 실사용 0건)
  OUTER_COAT        0건  (코드만 있음, 실사용 0건)
  BOTTOM_PANTS      0건  (코드만 있음, 실사용 0건)
  BOTTOM_SKIRT      0건  (코드만 있음, 실사용 0건)

--- DEFECT_TAXONOMY: 검증 여부 ---
  STAIN           trusted=True  [실측 검증됨] LAB-12 실측 양호 — 오염을 의류 특성 반영해 잘 분리
  DIRT_MARK       trusted=True  [실측 검증됨] LAB-12 실측 양호 — stain과 같은 프롬프트 계열
  HOLE_TEAR       trusted=False [실측 검증됨] LAB-46 실측 — SAM3가 뚜렷한 손상은 잘 잡지만 애매한 손상에서 구조적 개구부(암홀·넥라인·밑단·소매끝)를 구멍으로 오인하는 비유계 오탐이 반복 확인됨. 항상 SUSPECTED
  DISCOLORATION   trusted=False [미검증] 미검증 프롬프트 — SAM3 

**결론**: SAM3 자체의 "성능이 안 좋다"보다는 **범용 zero-shot 모델을 의류 도메인에 프롬프트만
바꿔 적용한 것**이 근본 원인이다 — 학습 데이터에 의류 하자 개념이 없으니, stain처럼 일반적인
"오염"과 개념이 가까운 유형은 잘 잡고(LAB-12), hole_tear처럼 "의류 고유의 정상 구조(개구부)"와
헷갈릴 수 있는 유형은 못 구분한다. `discoloration`/`pilling`/`fray`는 성능이 나쁜지 좋은지조차
모른다(**미검증**) — "안 좋다"고 단정할 근거도 없다.

**보완 방법(우선순위순)**:

1. **라벨셋 확보가 선행 조건** — 지금 6개 유형 중 3개가 미검증인 이유가 "검증할 라벨이 없어서"다.
   B절에서 제안한 셀러 피드백 루프가 여기서도 재사용된다.
2. **의류 특화 파인튜닝**(`garment-mask-finetune-check` 스파이크가 이 방향을 이미 탐색했다) —
   라벨셋이 쌓이면 SAM3 전체를 재학습하는 대신 LoRA 등 경량 파인튜닝으로 "의류의 정상 개구부"
   개념을 주입하는 게 비용 대비 합리적이다. `pilling`/`fray`처럼 SAM3가 원래 잘 못 볼 수도
   있는(질감형) 유형은 파인튜닝보다 **다른 모델(질감 분류기)을 보조로 붙이는 쪽**이 나을 수 있다.
3. **규칙 기반 보완(지금 하는 것)** — B절 제안처럼 부위별 스코프, 벤나인 억제 프롬프트 확장 등
   모델을 안 바꾸고 정책으로 오탐을 줄이는 저비용 대응. 라벨셋이 쌓이기 전까지의 임시방편.
4. **미검증 3종은 우선 스코프에서 빼는 것도 옵션** — 검증 안 된 걸 프로덕션에 계속 노출하는
   대신, 검증 전까지는 `needs_seller_review=True`로 고정해 "참고용"이라고 명확히 하는 지금
   방식이 최소한 안전하다(실제로 이미 그렇게 돼 있다).

## D. VLM 과다 호출 방지를 위한 정책 재검토

**현재 정책**(`quality_vlm/describer.py:attach_descriptions`): `status == DETECTED`인 finding만
VLM 설명을 만든다. SUSPECTED는 애초에 describer를 안 부른다 — CH011 계획이 명시한 v1 범위
제한(호출 수를 줄이기 위함)이다.

**비용 산정**: 실제 로그에서 DETECTED/SUSPECTED 비율과, description이 실제로 채워진 비율을
본다.

In [ ]:
detected = 0
suspected = 0
detected_with_desc = 0
jobs_with_desc_flag = 0

for job in SAM3_JOBS:
    findings = job.get("findings", [])
    for f in findings:
        if f["status"] == "DETECTED":
            detected += 1
            if f.get("description"):
                detected_with_desc += 1
        else:
            suspected += 1

print(f"실모델 job {len(SAM3_JOBS)}건 기준")
print(f"  DETECTED   {detected}건 (그중 description 채워짐 {detected_with_desc}건)")
print(f"  SUSPECTED  {suspected}건 (지금 정책상 전부 description 없음)")
print()
print("만약 SUSPECTED까지 전부 설명 대상으로 넓히면:")
print(f"  VLM 호출 수: {detected_with_desc} -> {detected_with_desc + suspected} (호출당 CH011 기준 수 초)")
print(f"  즉 현재 대비 최대 {suspected}건 추가 호출 — job 1건 평균 findings {len(SAM3_JOBS) and (detected+suspected)/len(SAM3_JOBS):.2f}건 감안하면 워밍업 후에도 latency가 findings 수에 비례해 늘어난다"
)


실모델 job 17건 기준
  DETECTED   23건 (그중 description 채워짐 2건)
  SUSPECTED  41건 (지금 정책상 전부 description 없음)

만약 SUSPECTED까지 전부 설명 대상으로 넓히면:
  VLM 호출 수: 2 -> 43 (호출당 CH011 기준 수 초)
  즉 현재 대비 최대 41건 추가 호출 — job 1건 평균 findings 3.76건 감안하면 워밍업 후에도 latency가 findings 수에 비례해 늘어난다


**읽는 법**: `description`이 실제로 채워진 건 23건의 DETECTED 중 2건뿐이다 — 나머지 21건은
`include_descriptions=false`로 요청된 job이었다는 뜻(옵트인이라 대부분 꺼져 있었음). 즉 지금도
실제 VLM 호출은 거의 안 일어나고 있다 — SUSPECTED까지 넓히는 논의 자체가 아직 "잠재적으로
얼마나 늘어날 수 있는가"를 보는 단계다.

**정책 옵션**:

1. **현재 유지(DETECTED만)** — 가장 안전하지만, SUSPECTED 하자(예: hole_tear)에 대해 FE가
   "설명이 아직 제공되지 않습니다"만 보여줘 사용자 경험상 어색하다(이번 대화에서 실제로 나온
   질문).
2. **`needs_seller_review=True`인 SUSPECTED만 확장** — hole_tear/discoloration/fray/pilling이
   전부 `needs_seller_review=True`라 사실상 거의 모든 SUSPECTED가 대상이 된다(위 41건 거의
   전부). 호출 수 절감 효과가 크지 않다 — **사실상 옵션 3과 비슷한 수준의 비용**이 든다.
3. **job당 설명 호출 개수 상한(budget)을 둔다** — confidence·severity 기준으로 상위 N개만
   설명하고 나머지는 "설명 생략(상한 도달)"로 명시한다. 지금처럼 "아직 제공 안 됨"이라는 모호한
   문구 대신 **왜 없는지 이유가 다른 문구**(SUSPECTED라서 vs 상한 초과라서)로 FE에 전달할 수
   있어야 한다 — 이건 응답 스키마에 필드가 하나 늘어야 하니 BE와 조율 필요.
4. **B절 제안(부위 스코프)이 통과되면 DETECTED 자체가 늘어나** 이 문제가 부분적으로 완화된다 —
   지금 "설명 없음"으로 보이는 CHEST_CENTER hole_tear 10건이 DETECTED로 승격되면 자동으로 설명
   대상이 되기 때문이다. B와 D는 독립적이지 않다.

**권장**: 지금 이 자리에서 정책을 못 박기보다, **B절 제안(부위 스코프)을 먼저 검증·적용**하고
남는 진짜 SUSPECTED(구조적 개구부 겹침 등 확실히 애매한 것들)에 대해서만 옵션 3(상한)을
검토하는 순서가 합리적이다.

## E. 탐지결과·bbox(FE) 전달용 그룹핑 정책

**현재 계약**(`AI/app/quality/schemas.py`): `findings`는 **평면(flat) 리스트**다. `image_index`/
`section`/`defect_type`가 각 finding에 붙어 있어 FE가 원하는 대로 재그룹핑할 수는 있지만, 그
그룹핑 로직을 AI/BE/FE 중 누가 갖고 있을지가 아직 안 정해졌다. `image_summaries`(이미지별 요약
문장)와 `section_scores`(이미지×부위 점수)는 이미 부분 그룹핑을 제공하고 있어 참고가 된다.

아래는 **AI가 순수 로직으로 만들 수 있는** 3가지 그룹핑 후보를 실제 로그 데이터(product 81)로
시연한 것 — 실제 계약을 바꾸는 게 아니라, BE/FE와 그룹핑 방식을 논의할 때 구체적인 모양을
보여주기 위한 프로토타입이다.

In [ ]:
from collections import defaultdict

# 실제 프로덕션 job(product_id=81) — 이미지 2장, findings 3건(오염 1 + 구멍·찢어짐 2)
sample_job = next(j for j in SAM3_JOBS if j.get("product_id") == 81)
findings = sample_job["findings"]


def group_by_image(findings):
    """이미지 탭 UI에 맞는 모양 — FE가 지금 image_index로 이미지를 넘기고 있다면 가장 자연스럽다."""
    out = defaultdict(list)
    for f in findings:
        out[f["image_index"]].append(f["key"])
    return dict(out)


def group_by_section(findings):
    """부위별 리스트 UI(예: '가슴 중앙 - 2건') — 판매자가 부위 단위로 확인/소명하기 좋다."""
    out = defaultdict(list)
    for f in findings:
        out[f["section"]].append(f["key"])
    return dict(out)


def group_by_defect_type(findings):
    """하자 유형별 리스트 UI(예: '오염 1건, 구멍·찢어짐 2건') — 등급/감점 근거를 유형별로 보여줄 때 맞다."""
    out = defaultdict(list)
    for f in findings:
        out[f["defect_type"]].append(f["key"])
    return dict(out)


print("이미지별:", group_by_image(findings))
print("부위별:  ", group_by_section(findings))
print("유형별:  ", group_by_defect_type(findings))


이미지별: {0: ['stain_0_5_1'], 1: ['hole_tear_1_5_1', 'hole_tear_1_5_2']}
부위별:   {'CHEST_CENTER': ['stain_0_5_1', 'hole_tear_1_5_1', 'hole_tear_1_5_2']}
유형별:   {'STAIN': ['stain_0_5_1'], 'HOLE_TEAR': ['hole_tear_1_5_1', 'hole_tear_1_5_2']}


**흥미로운 부산물**: 이 sample_job은 이미지 2장 다 `CHEST_CENTER`에서만 하자가 잡혔다 — 즉
"부위별" 그룹핑을 쓰면 이미지가 여러 장이어도 부위 하나로 뭉쳐지는 경우가 실제로 있다. 이건
장단점이 갈린다:

- **이미지별**이 FE 구현엔 제일 쉽다(어차피 사진마다 오버레이를 그려야 하니 image_index가
  1차 키인 게 자연스럽다). annotated_images/attention_maps도 이미 image_index 기준이라 일관됨.
- **부위별**은 "같은 부위, 다른 사진"이 하나로 뭉쳐 판매자가 "가슴 중앙에 문제가 있다"는 걸
  한눈에 보기엔 좋지만, 어느 사진의 어느 좌표인지 알려면 다시 `image_index`/`bbox`를 봐야 해서
  이미지별보다 정보가 한 단계 더 필요하다.
- **유형별**은 감점 근거("오염 1건, 구멍·찢어짐 2건 — 등급 A")를 보여주는 요약 UI에 맞고, 이미
  `summary`/`image_summaries` 필드가 텍스트로는 유형+부위를 섞어 제공하고 있어 구조화된 버전이
  필요하면 이 그룹핑이 후보가 된다.

**권장**: 계약을 당장 바꾸지 말고, 위 3가지 프로토타입을 FE 요구사항(어떤 화면에 먼저 쓸지)과
맞춰본 뒤 필요한 쪽 하나만 `app/quality/schemas.py`에 파생 필드로 추가하는 게 낫다 — 세 가지를
다 얹으면 응답이 불필요하게 무거워진다. 확정되면 `docs/plans/AI/`에 계획 문서로 남기고 진행.

## 결론 및 후속 작업

- **B(부위별 신뢰 스코프)**가 가장 실행 가능성이 높다 — 실제 로그로 근거가 확인됐고(`CHEST_CENTER`
  10건이 즉시 후보), 코드 변경 범위도 `DEFECT_TAXONOMY`/`_status_of` 정도로 작다. 다만 **적용
  전에 계획 문서(`docs/plans/AI/`)를 먼저 쓴다** — 루트 CLAUDE.md 규칙.
- **A(프롬프트 실험)**는 B를 뒷받침하는 근거를 더 쌓을 수 있다(특히 `HEM_*`처럼 threshold를
  넘는데도 부위상 못 믿는 경우가 benign 프롬프트를 더 강하게 만들면 해결되는지) — GPU 워커에서
  실행 필요.
- **C(탐지 범위)**는 즉시 실행 항목이 없다 — "미검증 3종은 지금처럼 review 필수로 유지"가 이미
  현재 상태와 같다. 다음에 라벨셋이 생기면 이 3종부터 우선 검증한다.
- **D(VLM 정책)**는 B의 결과에 종속적이다 — B가 먼저 적용되고 남는 진짜 SUSPECTED 규모를 다시
  본 뒤에 상한(옵션 3) 여부를 정한다.
- **E(그룹핑)**는 AI 단독 결정이 아니라 FE 요구사항이 필요하다 — 필요해지면 `docs/requests/FE/`
  또는 계획 문서로 조율.
- 셀러 피드백 루프(B/C에서 반복 언급된 "라벨셋 없음"의 근본 해법)는 BE 저장 스키마가 관여하니
  별도로 `docs/requests/BE/`에 요청 문서화가 필요하다.
